# Abia SARIMA Malaria Intervention-Mix Model

Run this notebook from the project root. The source files are modularised under `src/` so every assumption can be changed without rewriting the full workflow.

**Date alignment:** The 64 observations map exactly to the 64 monthly periods from **January 2021 through April 2026**. No month is inserted or treated as missing. The 24-month forecast covers **May 2026 through April 2028**.

## Purpose

Attach referenced intervention costs, calculate net cost per case averted and generate 17 separate four-panel LGA reports.

In [ ]:
from pathlib import Path
import sys, pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
%cd $PROJECT_ROOT

In [ ]:
from src.data import load_all_lgas
from src.intervention import evaluate_scenarios
from src.costing import add_cost_effectiveness, incremental_cea, build_government_recommendations
from src.plotting import plot_all_lgas
from src.config import OUTPUT_DIR, COST_LIBRARY
long_df, meta, validation = load_all_lgas()
group_forecast = pd.read_csv(OUTPUT_DIR / "forecasts" / "group_SARIMA_forecasts_24m.csv", parse_dates=["date"])
total_forecast = pd.read_csv(OUTPUT_DIR / "forecasts" / "LGA_total_SARIMA_forecasts_24m.csv", parse_dates=["date"])

In [ ]:
pd.DataFrame(COST_LIBRARY).T[["base_usd_2024", "low_usd_2024", "high_usd_2024", "unit", "source"]]

In [ ]:
monthly_scenarios, cumulative_scenarios = evaluate_scenarios(group_forecast, meta)
monthly_costs, results = add_cost_effectiveness(monthly_scenarios, cumulative_scenarios, meta)
incremental = incremental_cea(results)
recommendations = build_government_recommendations(results)
recommendations

In [ ]:
paths = plot_all_lgas(long_df, total_forecast, results, meta)
print(f"Generated {len(paths)} LGA reports")

In [ ]:
results.to_csv(OUTPUT_DIR / "tables" / "LGA_scenario_cost_effectiveness.csv", index=False)
incremental.to_csv(OUTPUT_DIR / "tables" / "incremental_cost_effectiveness.csv", index=False)
recommendations.to_csv(OUTPUT_DIR / "tables" / "government_recommendations.csv", index=False)

### Cost evidence

The defaults come from malaria costing systematic reviews, the Nigeria PBO-net economic evaluation, vaccine-delivery costing studies and LSM costing studies. Source URLs and price-year metadata are retained in `src/config.py` and `REFERENCES.md`. Replace them with Abia procurement and programme-delivery costs when available.